In [3]:
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [ ]:
import csv
import glob
import os

col_names = [
    'col0','col1','col2','row_num','unternehmen','kd_nr','typ_amed','typ_asi',
    'arzt','fasi','betreuungsart','direkte_leistung','mitarbeiter','bg',
    'vertragsbeginn','start_aktuelle_periode','vertragsstunden','davon_amed',
    'davon_asi','davon_apsy','zusaetzl_amed','zusaetzl_asi','zusaetzl_apsy',
    'vertragsstunden_direkt','amed_direkt','asi_direkt','apsy_direkt',
    'stunden_erbracht','amed_erbracht','asi_erbracht','apsy_erbracht',
    'fahrtzeit_erbracht','stunden_offen','amed_offen','asi_offen','apsy_offen',
    'beleistung_gesamt','beleistung_amed','beleistung_asi',
    'gesamt','spezifisch_amed','spezifisch_asi','col42'
]

all_rows = []
csv_files = sorted(glob.glob('basic_care_details_data/Stundenerfassung Dashboard_*.csv'))
for f in csv_files:
    city = os.path.basename(f).split('(')[1].split(')')[0]
    with open(f, encoding='latin-1') as fh:
        reader = list(csv.reader(fh, delimiter=';'))
        for r in reader[5:]:
            if len(r) >= 42 and r[3].strip().isdigit():
                all_rows.append([city] + r[:43])

duck.sql('CREATE OR REPLACE TABLE raw_details (standort VARCHAR, ' + ', '.join([f'{c} VARCHAR' for c in col_names]) + ')')
duck.executemany('INSERT INTO raw_details VALUES (' + ','.join(['?'] * 44) + ')', all_rows)

print(f"Loaded {len(all_rows)} rows from {len(csv_files)} files")
duck.sql("SELECT standort, unternehmen, kd_nr, betreuungsart, vertragsstunden FROM raw_details LIMIT 5")

In [ ]:
def parse_german_num(col):
    return f"TRY_CAST(replace(replace(replace(trim({col}), ' ', ''), ',', '.'), chr(160), '') AS DOUBLE)"

german_months = {
    'Jan': '01', 'Feb': '02', 'Mrz': '03', 'Apr': '04',
    'Mai': '05', 'Jun': '06', 'Jul': '07', 'Aug': '08',
    'Sep': '09', 'Okt': '10', 'Nov': '11', 'Dez': '12'
}

def german_date_expr(col):
    """Parse German date like '1/ Apr 26' → DATE, returns NULL for invalid values like #VALEUR!"""
    expr = f"trim({col})"
    for de, num in german_months.items():
        expr = f"replace({expr}, '{de}', '{num}')"
    return f"CASE WHEN trim({col}) NOT LIKE '#%' AND trim({col}) != '' AND trim({col}) != '-' THEN TRY_CAST(strptime(trim({expr}), '%d/ %m %y') AS DATE) ELSE NULL END"

duck.sql(f"""
    CREATE OR REPLACE TABLE basic_care_details AS
    SELECT
        left(trim(kd_nr), 9) as mother_client_id,
        first(trim(unternehmen)) as firm_name,
        first(trim(betreuungsart)) as offer_type,
        ROUND(SUM({parse_german_num('vertragsstunden')}), 2) as sold_hours,
        ROUND(SUM({parse_german_num('davon_amed')}) / NULLIF(SUM({parse_german_num('vertragsstunden')}), 0) * 100)::int as doctor_ratio,
        ROUND(SUM({parse_german_num('davon_asi')}) / NULLIF(SUM({parse_german_num('vertragsstunden')}), 0) * 100)::int as preventer_ratio,
        TRY_CAST(replace(trim(first(direkte_leistung)), '%', '') AS INT) as direct_time_ratio,
        100 - TRY_CAST(replace(trim(first(direkte_leistung)), '%', '') AS INT) as indirect_time_ratio,
        SUM(TRY_CAST(trim(mitarbeiter) AS INT))::int as mitarbeiter,
        MIN({german_date_expr('vertragsbeginn')}) as contract_start_date,
        MIN({german_date_expr('start_aktuelle_periode')}) as first_renewal_date,
        12 as renewal_frequency_in_months,
        first(trim(typ_amed)) FILTER (WHERE trim(typ_amed) != '') as typ_amed,
        first(trim(typ_asi)) FILTER (WHERE trim(typ_asi) != '') as typ_asi
    FROM raw_details
    WHERE trim(kd_nr) IS NOT NULL AND trim(kd_nr) != ''
    GROUP BY left(trim(kd_nr), 9)
    ORDER BY mother_client_id
""")

duck.sql("SELECT count(*) as unique_clients FROM basic_care_details")

In [ ]:
# Preview aggregated data
duck.sql("SELECT * FROM basic_care_details LIMIT 10")

In [ ]:
# Write to PostgreSQL: bas_firms.basic_care_details
duck.sql("""
    DROP TABLE IF EXISTS pg.bas_firms.basic_care_details;
    CREATE TABLE pg.bas_firms.basic_care_details AS
    SELECT * FROM basic_care_details;
""")

duck.sql("SELECT count(*) as rows_in_db FROM pg.bas_firms.basic_care_details")